In [ ]:
# from huggingface_hub import notebook_login

# notebook_login()

In [ ]:
!pip install -q diffusers --upgrade

In [ ]:
!pip install -q invisible_watermark transformers accelerate safetensors gradio

In [ ]:
import torch
from diffusers import StableDiffusionXLImg2ImgPipeline, StableDiffusionImg2ImgPipeline
from diffusers.utils import load_image
import gradio as gr

#runwayml/stable-diffusion-v1-5
#stabilityai/stable-diffusion-xl-refiner-1.0
device = "cuda" if torch.cuda.is_available() else "cpu"

# pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
#     "stabilityai/stable-diffusion-xl-refiner-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True
# ).to(device)

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to(device)

pipe.enable_attention_slicing()

In [ ]:
#pipe.unet = torch.compile(pipe.unet, mode="reduce-overhead", fullgraph=True)

In [ ]:
PROMPT_OPTIONS = {
    "Dents, Bend, Inclination":
        "A high-resolution, ultra-realistic underwater ROV inspection of a weathered yellow subsea equipment module with visible rust, corroded surfaces, hydraulic cables, and metal fixtures. Dark deep-sea background, lit by artificial subsea lights, detailed marine residue, aged industrial machinery, realistic textures, authentic ROV camera footage style, cinematic.",

    "Corroded | Pipeline fading":
        "An underwater remotely operated vehicle (ROV) inspecting a yellow metallic subsea unit with hydraulic hoses and mechanical parts, deep sea environment, dark ocean background, industrial marine equipment, inspection footage with telemetry overlay, realistic lighting, high detail, captured from ROV camera",

    "Corroded | Coral-covered industrial pipe":
        "An old industrial pipe overgrown with coral and sponges, rust bleeding into the water, quiet deep ocean ambiance with faint beams of surface light",
}


#negative_prompt = "3D render, 3D cartoon, stylized, painting, illustration, CGI, low detail, low resolution, plastic textures"
negative_prompt = "3D cartoon, illustration, CGI, overly clean, smooth surfaces, fantasy, bright daylight, humans, abstract, low detail, low resolution"


STRENGTH = 0.75
GUIDANCE_SCALE = 7.5

examples = [
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/Dent_on_roof_of_UTA.jpg',
        'Dents, Bend, Inclination',
    ],
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/Bent_top_plate_of_VCS.jpg',
        'Dents, Bend, Inclination',
    ],
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/Excessive_Bend_movement_on_Riser static_section.jpg',
        'Dents, Bend, Inclination',
    ],
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/Inclination_of_PLET.jpg',
        'Dents, Bend, Inclination',
    ],
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/Inclination_of_UTH.jpg',
        'Dents, Bend, Inclination',
    ],
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/Measurement_bar_bent.jpg',
        'Dents, Bend, Inclination',
    ],
    [
        'https://raw.githubusercontent.com/docty/image-creation/refs/heads/main/Measurement_bar_bent2.jpg',
        'Dents, Bend, Inclination',
    ],
    
     
]

def generate_image(init_image, prompt_choice):
    init_image = init_image.convert("RGB").resize((768, 512))
    prompt = PROMPT_OPTIONS[prompt_choice]

    result = pipe(
        prompt=prompt,
        image=init_image,
        strength=STRENGTH,
        guidance_scale=GUIDANCE_SCALE,
        negative_prompt=negative_prompt
    ).images[0]

    return result

with gr.Blocks(css="""
    .gr-block { padding: 1rem !important; }
    .gr-button { width: 100%; font-weight: bold; }
    .gr-image { border-radius: 12px; }
""") as demo:

    gr.Markdown("""
        <h1 style='text-align: center; color: #333;'> Image Generator</h1>
        <p style='text-align: center; max-width: 700px; margin: auto; color: #555;'>
            Upload an initial image and choose a corrosion style prompt to generate an image or choose an Example from below.
        </p>
    """)

    # Dropdown and Button on Top
    with gr.Row():
        prompt_choice = gr.Dropdown(
            choices=list(PROMPT_OPTIONS.keys()),
            label="Select Anomalies Type",
            value="Dents, Bend, Inclination"
        )
        generate_btn = gr.Button("Generate Image")

    # Upload and Generated Image Below
    with gr.Row():
        with gr.Column(scale=1):
            init_image = gr.Image(
                label="Upload Your Image",
                type="pil",
                height=400,
                width=400
            )
        with gr.Column(scale=1):
            output_image = gr.Image(label="Generated Image", height=300)

    generate_btn.click(
        fn=generate_image,
        inputs=[init_image, prompt_choice],
        outputs=output_image
    )

    gr.Examples(examples=examples, inputs=[init_image, prompt_choice])

demo.launch(share=True)
